# Объединение датасетов и пересборка BPE-токенизатора

Ноутбук выполняет:
1. Обработку новых видео (pt16–pt21) из `Dataset_VSR/` → извлечение губ → `.pkl`
2. Сбор текстового корпуса из существующих `.pkl` + транскрипций новых видео
3. Обучение нового BPE-токенизатора на объединённом корпусе
4. Ретокенизацию всех `.pkl` (старых + новых) единым токенизатором
5. Обновление конфига модели

## Параметры

In [1]:
from pathlib import Path
import re

# ── Пути ──
# Существующий датасет (pkl с GPT-2 input_ids)
# OLD_DATA_PATH     = Path("D:/Datasets/Dataset_720_proc")
OLD_DATA_PATH     = Path("C:/Docs/GitHub/Datasets/Dataset_720_proc")

# Новые видео (mp4 + csv)
# NEW_RAW_PATH      = Path("D:/Datasets/Dataset_VSR")
NEW_RAW_PATH      = Path("C:/Docs/GitHub/Datasets/Dataset_VSR")
NEW_VIDEO_PATH    = NEW_RAW_PATH / "video"
NEW_TRANSCRIPT_PATH = NEW_RAW_PATH / "transcript"

# Куда сохранить обработанные новые видео (pkl)
# NEW_PROCESSED_PATH = Path("D:/Datasets/Dataset_VSR_proc")
NEW_PROCESSED_PATH = Path("C:/Docs/GitHub/Datasets/Dataset_VSR_proc")

# Итоговый объединённый датасет (BPE токенизация)
# MERGED_DATA_PATH  = Path("D:/Datasets/Dataset_merged_bpe")
MERGED_DATA_PATH  = Path("C:/Docs/GitHub/Dataset_merged_bpe")

# Токенизатор и конфиг
TOKENIZER_PATH    = Path("configs/bpe_tokenizer_merged")
DST_CONFIG        = Path("../python_mbconv/configs/model_config_mbconv_micro_merged.json")
SRC_CONFIG        = Path("../python_mbconv/configs/model_config_mbconv_micro.json")

# Какие клипы есть в старом датасете
OLD_CLIPS = [f"pt{i}" for i in range(1, 16)]  # pt1–pt15

# Какие клипы в новых видео
NEW_CLIPS = [f"pt{i}" for i in range(16, 22)]  # pt16–pt21

# BPE параметры
VOCAB_SIZE = 1200
BOS_TOKEN  = "<|bos|>"
EOS_TOKEN  = "<|eos|>"
PAD_TOKEN  = "<|pad|>"
UNK_TOKEN  = "<|unk|>"

# Функция очистки текста
def clean_text(text):
    """Убрать пунктуацию, lowercase, оставить апострофы."""
    return re.sub(r"[^\w\s']", '', text.lower()).strip()

print(f"Старый датасет:   {OLD_DATA_PATH}")
print(f"Новые видео:      {NEW_RAW_PATH}")
print(f"Итоговый датасет: {MERGED_DATA_PATH}")

Старый датасет:   C:\Docs\GitHub\Datasets\Dataset_720_proc
Новые видео:      C:\Docs\GitHub\Datasets\Dataset_VSR
Итоговый датасет: C:\Docs\GitHub\Dataset_merged_bpe


## Шаг 1. Обработка новых видео → pkl

Используем `LipReading2Preprocessor` из `find_face.py` для извлечения
области губ из каждого кадра и сохранения в `.pkl` формате.

In [2]:
# %pip install --upgrade opencv-python
# %pip install "numpy<2"
# %pip install dlib
# %pip install --force-reinstall dlib
# %pip install dlib==19.24.2
# %pip install dlib==19.24.6 --find-links https://github.com/z-mahmud22/Dlib_Windows_Python3.x/releases/tag/v19.24.6
# %pip install https://github.com/z-mahmud22/Dlib_Windows_Python3.x/releases/download/v19.24.6/dlib-19.24.6-cp311-cp311-win_amd64.whl


import sys
sys.path.insert(0, ".")

from find_face import LipReading2Preprocessor

preprocessor = LipReading2Preprocessor(
    raw_data_path=str(NEW_RAW_PATH),
    processed_data_path=str(NEW_PROCESSED_PATH),
)

print(f"Обработка видео из {NEW_RAW_PATH} ...")
preprocessor.process_videos()
print("Готово.")

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

c:\Users\m.vashkevich\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\m.vashkevich\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Лица не обнаружены
Обработано: pt19.mp4_55 -> 30 кадров
Обработано: pt20.mp4_0 -> 44 кадров
Обработано: pt20.mp4_1 -> 15 кадров
Обработано: pt20.mp4_2 -> 30 кадров
Обработано: pt20.mp4_3 -> 44 кадров
Обработано: pt20.mp4_4 -> 45 кадров
Обработано: pt20.mp4_5 -> 44 кадров
Обработано: pt20.mp4_6 -> 45 кадров
Обработано: pt20.mp4_7 -> 15 кадров
Обработано: pt20.mp4_8 -> 30 кадров
Обработано: pt20.mp4_9 -> 45 кадров
Лица не обнаружены
Лица не обнаружены
Лица не обнаружены
Обработано: pt20.mp4_10 -> 41 кадров
Обработано: pt20.mp4_11 -> 45 кадров
Обработано: pt20.mp4_12 -> 44 кадров
Обработано: pt20.mp4_13 -> 45 кадров
Обработано: pt20.mp4_14 -> 44 кадров
Обработано: pt20.mp4_15 -> 15 кадров
Обработано: pt20.mp4_16 -> 30 кадров
Обработано: pt20.mp4_17 -> 44 кадров
Обработано: pt20.mp4_18 -> 45 кадров
Обработано: pt20.mp4_19 -> 44 кадров
Обработано: pt20.mp4_20 -> 45 кадров
Обработано: pt20.mp4_21 -> 44 кадров
Обработано: pt20.mp4_22 -> 45 кадров
Обработано: pt20.mp4_23 -> 44 кадров
Обработан

 83%|████████▎ | 5/6 [3:46:44<33:02, 1982.24s/it]  

Лица не обнаружены
Обработано: pt20.mp4_61 -> 15 кадров
Обработано: pt21.mp4_0 -> 50 кадров
Обработано: pt21.mp4_1 -> 100 кадров
Обработано: pt21.mp4_2 -> 50 кадров
Обработано: pt21.mp4_3 -> 50 кадров
Обработано: pt21.mp4_4 -> 50 кадров
Обработано: pt21.mp4_5 -> 50 кадров
Обработано: pt21.mp4_6 -> 50 кадров
Обработано: pt21.mp4_7 -> 50 кадров
Обработано: pt21.mp4_8 -> 100 кадров
Обработано: pt21.mp4_9 -> 50 кадров
Обработано: pt21.mp4_10 -> 50 кадров
Обработано: pt21.mp4_11 -> 50 кадров
Обработано: pt21.mp4_12 -> 50 кадров
Обработано: pt21.mp4_13 -> 50 кадров
Обработано: pt21.mp4_14 -> 50 кадров
Обработано: pt21.mp4_15 -> 50 кадров
Обработано: pt21.mp4_16 -> 50 кадров
Обработано: pt21.mp4_17 -> 50 кадров
Обработано: pt21.mp4_18 -> 50 кадров
Обработано: pt21.mp4_19 -> 50 кадров
Обработано: pt21.mp4_20 -> 50 кадров
Обработано: pt21.mp4_21 -> 25 кадров
Обработано: pt21.mp4_22 -> 25 кадров
Обработано: pt21.mp4_23 -> 50 кадров
Обработано: pt21.mp4_24 -> 50 кадров
Обработано: pt21.mp4_25 -> 

100%|██████████| 6/6 [4:26:01<00:00, 2660.33s/it]

Лица не обнаружены
Предупреждение: Не удалось обработать pt21.mp4_94
Готово.


In [3]:
# Проверка: сколько pkl файлов создано
import pickle

new_pkl_count = 0
for clip in NEW_CLIPS:
    clip_dir = NEW_PROCESSED_PATH / clip
    if clip_dir.exists():
        files = list(clip_dir.glob("*.pkl"))
        new_pkl_count += len(files)
        print(f"  {clip}: {len(files)} файлов")
    else:
        print(f"  {clip}: папка не найдена")

print(f"\nВсего новых pkl: {new_pkl_count}")

# Проверяем формат первого файла
sample_pkl = next(NEW_PROCESSED_PATH.glob("**/*.pkl"))
with open(sample_pkl, "rb") as f:
    sample = pickle.load(f)
print(f"\nПример: {sample_pkl.name}")
print(f"  keys: {list(sample.keys())}")
print(f"  num_frames: {sample['num_frames']}")
print(f"  input_ids (GPT-2): {sample['input_ids'][:10]}...")
print(f"  frame shape: {sample['frames'][0].shape}")

  pt16: 50 файлов
  pt17: 36 файлов
  pt18: 30 файлов
  pt19: 56 файлов
  pt20: 62 файлов
  pt21: 93 файлов

Всего новых pkl: 327

Пример: pt16_0.pkl
  keys: ['frames', 'tokens', 'input_ids', 'num_frames']
  num_frames: 75
  input_ids (GPT-2): [10449, 345, 13, 317, 845, 5814, 7062, 284, 7823, 3576]...
  frame shape: (64, 96, 3)


## Шаг 2. Сбор объединённого текстового корпуса

Тексты из двух источников:
- Старый датасет: декодируем `input_ids` через GPT-2 → очищаем
- Новый датасет: декодируем `input_ids` через GPT-2 → очищаем

In [4]:
from transformers import AutoTokenizer

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")

corpus_texts = []   # все тексты для обучения BPE
all_files    = []   # (pkl_path, source, cleaned_text) для шага 4

# ── Старый датасет ──
old_count = 0
for clip_name in sorted(OLD_CLIPS):
    clip_dir = OLD_DATA_PATH / clip_name
    if not clip_dir.exists():
        continue
    for pkl_path in sorted(clip_dir.glob("*.pkl"),
                           key=lambda f: int(f.stem.split("_")[-1])):
        with open(pkl_path, "rb") as f:
            data = pickle.load(f)
        raw = gpt2_tokenizer.decode(data["input_ids"])
        text = clean_text(raw)
        if text:
            corpus_texts.append(text)
            all_files.append((pkl_path, "old", text))
            old_count += 1

# ── Новый датасет ──
new_count = 0
for clip_name in sorted(NEW_CLIPS):
    clip_dir = NEW_PROCESSED_PATH / clip_name
    if not clip_dir.exists():
        continue
    for pkl_path in sorted(clip_dir.glob("*.pkl"),
                           key=lambda f: int(f.stem.split("_")[-1])):
        with open(pkl_path, "rb") as f:
            data = pickle.load(f)
        raw = gpt2_tokenizer.decode(data["input_ids"])
        text = clean_text(raw)
        if text:
            corpus_texts.append(text)
            all_files.append((pkl_path, "new", text))
            new_count += 1

print(f"Файлов из старого датасета: {old_count}")
print(f"Файлов из нового датасета:  {new_count}")
print(f"Всего текстов для корпуса:  {len(corpus_texts)}")
print()
print("=== Примеры (старые) ===")
for t in corpus_texts[:3]:
    print(f"  {repr(t)}")
print("=== Примеры (новые) ===")
for t in corpus_texts[old_count:old_count+3]:
    print(f"  {repr(t)}")

Файлов из старого датасета: 861
Файлов из нового датасета:  327
Всего текстов для корпуса:  1188

=== Примеры (старые) ===
  'hello and a very warm welcome to bbc'
  'london coming up tonight'
  'good evening to you we start tonight'
=== Примеры (новые) ===
  'thank you a very warm welcome to bbc london'
  'coming up on the program'
  "they warn half the met's buildings face closure"


## Шаг 3. Анализ объединённого корпуса

In [5]:
from collections import Counter

all_text = " ".join(corpus_texts)
words    = re.findall(r"\b\w+\b", all_text)
word_freq = Counter(words)

print(f"Всего символов в корпусе:   {len(all_text):,}")
print(f"Всего слов (с повторами):   {len(words):,}")
print(f"Уникальных слов:            {len(word_freq):,}")
print()
print("Топ-20 слов:")
for word, cnt in word_freq.most_common(20):
    print(f"  {word!r:20s} {cnt}")

Всего символов в корпусе:   44,022
Всего слов (с повторами):   8,081
Уникальных слов:            2,049

Топ-20 слов:
  'the'                386
  'to'                 225
  'and'                213
  'a'                  205
  's'                  168
  'of'                 160
  'in'                 141
  'for'                106
  'london'             103
  'on'                 100
  'you'                87
  'it'                 79
  'is'                 72
  'has'                72
  'with'               70
  'from'               70
  'been'               70
  'that'               69
  'now'                69
  'our'                60


## Шаг 4. Обучение BPE-токенизатора на объединённом корпусе

In [6]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(BPE(unk_token=UNK_TOKEN))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=[PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN],
    min_frequency=1,
    show_progress=True,
)

tokenizer.train_from_iterator(corpus_texts, trainer=trainer)

PAD_TOKEN_ID = tokenizer.token_to_id(PAD_TOKEN)
BOS_TOKEN_ID = tokenizer.token_to_id(BOS_TOKEN)
EOS_TOKEN_ID = tokenizer.token_to_id(EOS_TOKEN)
actual_vocab = tokenizer.get_vocab_size()

print(f"Реальный vocab_size: {actual_vocab}")
print(f"PAD={PAD_TOKEN_ID}, BOS={BOS_TOKEN_ID}, EOS={EOS_TOKEN_ID}")

# Сохраняем
TOKENIZER_PATH.mkdir(parents=True, exist_ok=True)
tokenizer.save(str(TOKENIZER_PATH / "tokenizer.json"))
print(f"Сохранён: {TOKENIZER_PATH / 'tokenizer.json'}")

# Проверка
for t in corpus_texts[:3]:
    enc = tokenizer.encode(t)
    print(f"  {repr(t[:50])} → {enc.ids}")

Реальный vocab_size: 1200
PAD=0, BOS=1, EOS=2
Сохранён: configs\bpe_tokenizer_merged\tokenizer.json
  'hello and a very warm welcome to bbc' → [663, 56, 15, 187, 649, 379, 52, 125]
  'london coming up tonight' → [80, 254, 148, 237]
  'good evening to you we start tonight' → [311, 136, 52, 76, 85, 627, 237]


## Шаг 5. Сохранение объединённого датасета с новыми input_ids

Все pkl (старые + новые) копируются в `MERGED_DATA_PATH`
с ретокенизированными `input_ids`.

In [7]:
converted = 0
skipped   = 0

for pkl_src, source, text in all_files:
    # Определяем имя клипа (ptN) из пути
    clip_name = pkl_src.parent.name   # e.g. "pt1", "pt16"
    file_name = pkl_src.name          # e.g. "pt1_0.pkl"

    pkl_dst = MERGED_DATA_PATH / clip_name / file_name
    pkl_dst.parent.mkdir(parents=True, exist_ok=True)

    with open(pkl_src, "rb") as f:
        data = pickle.load(f)

    new_ids = tokenizer.encode(text).ids
    if len(new_ids) == 0:
        print(f"[!] Пустая токенизация: {pkl_src.name}, пропуск")
        skipped += 1
        continue

    data["input_ids"] = new_ids

    with open(pkl_dst, "wb") as f:
        pickle.dump(data, f)

    converted += 1

print(f"Конвертировано: {converted} файлов")
print(f"Пропущено:      {skipped} файлов")
print(f"Датасет:        {MERGED_DATA_PATH}")
print()

# Статистика по клипам
for clip_dir in sorted(MERGED_DATA_PATH.iterdir()):
    if clip_dir.is_dir():
        n = len(list(clip_dir.glob("*.pkl")))
        print(f"  {clip_dir.name}: {n} файлов")

Конвертировано: 1188 файлов
Пропущено:      0 файлов
Датасет:        C:\Docs\GitHub\Dataset_merged_bpe

  pt1: 62 файлов
  pt11: 60 файлов
  pt12: 70 файлов
  pt13: 67 файлов
  pt14: 84 файлов
  pt15: 52 файлов
  pt16: 50 файлов
  pt17: 36 файлов
  pt18: 30 файлов
  pt19: 56 файлов
  pt2: 73 файлов
  pt20: 62 файлов
  pt21: 93 файлов
  pt3: 71 файлов
  pt4: 62 файлов
  pt5: 68 файлов
  pt6: 22 файлов
  pt7: 50 файлов
  pt8: 63 файлов
  pt9: 57 файлов


## Шаг 6. Обновление конфига модели

In [8]:
import json

with open(SRC_CONFIG, "r", encoding="utf-8") as f:
    config = json.load(f)

old_vocab = config["vocab_size"]
config["vocab_size"] = actual_vocab

with open(DST_CONFIG, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"Конфиг: {DST_CONFIG}")
print(f"  vocab_size: {old_vocab} → {actual_vocab}")
print()
print(json.dumps(config, indent=2, ensure_ascii=False))

Конфиг: ..\python_mbconv\configs\model_config_mbconv_micro_merged.json
  vocab_size: 1200 → 1200

{
  "vocab_size": 1200,
  "d_model": 48,
  "nhead": 2,
  "num_layers": 2,
  "num_encoder_layers": 2,
  "num_decoder_layers": 1,
  "dim_feedforward": 96,
  "dropout": 0.5,
  "max_frames": 40,
  "max_tokens": 12,
  "encoder": {
    "stem_channels": 24,
    "drop_path_rate": 0.15,
    "mbconv_blocks": [
      {
        "in_channels": 24,
        "out_channels": 48,
        "kernel_size": 3,
        "stride": 2,
        "expand_ratio": 2,
        "se_ratio": 0.25,
        "use_se": true,
        "_comment": "Downsample 32x48 → 16x24, ~6K params"
      },
      {
        "in_channels": 48,
        "out_channels": 64,
        "kernel_size": 3,
        "stride": 2,
        "expand_ratio": 2,
        "se_ratio": 0.25,
        "use_se": true,
        "_comment": "Downsample 16x24 → 8x12, ~18K params"
      }
    ]
  }
}


## Шаг 7. Проверка объединённого датасета

In [9]:
import numpy as np

# Статистика длин токенов
token_lengths = []
for pkl_path, _, text in all_files:
    token_lengths.append(len(tokenizer.encode(text).ids))

print(f"Длина последовательности токенов:")
print(f"  min:    {min(token_lengths)}")
print(f"  max:    {max(token_lengths)}")
print(f"  median: {int(np.median(token_lengths))}")
print(f"  mean:   {np.mean(token_lengths):.1f}")
print()

# Проверка нескольких файлов
print("=== Примеры из объединённого датасета ===")
for clip in ["pt1", "pt16"]:
    clip_dir = MERGED_DATA_PATH / clip
    if not clip_dir.exists():
        continue
    sample_path = sorted(clip_dir.glob("*.pkl"))[0]
    with open(sample_path, "rb") as f:
        s = pickle.load(f)
    decoded = tokenizer.decode(s["input_ids"])
    print(f"  {clip}/{sample_path.name}: frames={s['num_frames']}, "
          f"ids={s['input_ids'][:8]}..., text={repr(decoded[:60])}")

Длина последовательности токенов:
  min:    1
  max:    24
  median: 10
  mean:   9.6

=== Примеры из объединённого датасета ===
  pt1/pt1_0.pkl: frames=50, ids=[663, 56, 15, 187, 649, 379, 52, 125]..., text='hello and a very warm welcome to bbc'
  pt16/pt16_0.pkl: frames=75, ids=[642, 76, 15, 187, 649, 379, 52, 125]..., text='thank you a very warm welcome to bbc london'


## Итог

После выполнения:
- Объединённый датасет: `D:/Datasets/Dataset_merged_bpe/` (pt1–pt21)
- Токенизатор: `configs/bpe_tokenizer_merged/tokenizer.json`
- Конфиг: `python_mbconv/configs/model_config_mbconv_micro_merged.json`

**Для обучения в `train_bpe.ipynb` изменить:**
```python
DATA_PATH         = "D:/Datasets/Dataset_merged_bpe"
TOKENIZER_PATH    = "../python/configs/bpe_tokenizer_merged/tokenizer.json"
MODEL_CONFIG_PATH = "configs/model_config_mbconv_micro_merged.json"

# Обновить сплиты:
TRAIN_CLIPS = [f"pt{i}" for i in range(1, 18)]   # pt1–pt17 (расширенный train)
VAL_CLIPS   = [f"pt{i}" for i in range(18, 20)]   # pt18–pt19
TEST_CLIPS  = [f"pt{i}" for i in range(20, 22)]   # pt20–pt21
```